# Домашняя работа 1. Своя таблица: утечка, пропуски и кодирование

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| К лабораторной | занятие 1 — Инструменты, данные и первый ориентир |
| Опора | материал семинара 1 и лекций до него |
| Ожидаемое время | 3–4 часа |
| Данные | **ваша индивидуальная таблица** (по ФИО) |

На занятии мы разбирали «Титаник» — учебную таблицу, одинаковую у всех. Дома вы получаете свою собственную: она порождена по вашему ФИО и ни у кого в группе не повторяется. Дефекты в ней те же по природе, что и в «Титанике», но расположены иначе, и найти их придётся самостоятельно.\n\nТри задачи: разведка с поиском утечки, заполнение пропусков с учётом механизма и своя реализация One-Hot кодирования.

> **Чем это отличается от занятия.** На семинаре данные были учебные и общие —
> так удобно разбирать. Дома данные ваши: таблица порождается по ФИО, и ни у
> кого в группе она не повторяется. Приёмы те же, числа другие — поэтому
> отвечать придётся за свои числа, а не за преподавательские.


## Как устроена работа

Работа делится на две части, и делятся они по назначению, а не по сложности.

**Обязательная часть — допуск.** Без неё работа не принимается: это тот минимум,
без которого занятие считается неусвоенным. Здесь всегда есть хотя бы одна
реализация «с нуля», сверенная с `scikit-learn` численно.

**Часть на оценку** (помечена значком ★). Она не нужна для допуска — но балл
за работу выставляется именно по ней, и каждый выполненный пункт идёт в зачёт
отдельно. Браться стоит даже за один пункт: это лучше, чем не браться вовсе.

## Что нужно сдать

Заполненный ноутбук, в котором:

1. выполнены все ячейки с `# TODO` в обязательной части, код исполняется сверху
   вниз без ошибок в свежем ядре (Kernel → Restart & Run All);
2. под каждым заданием заполнена ячейка **Вывод** — своими словами,
   со ссылкой на полученные числа;
3. графики подписаны: заголовок, оси, легенда;
4. в ячейке варианта вписано ваше ФИО.

> Списывание видно сразу: у каждого студента свой датасет и свой набор методов.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from variants import make_table

## Индивидуальный вариант

Впишите своё ФИО (или почту) в переменную `STUDENT` — вариант вычисляется детерминированно, при повторном запуске он тот же самый.

Регистр, лишние пробелы и написание «ё»/«е» роли не играют. Если ФИО вписано неверно, вариант будет чужим — проверьте вывод ячейки.

In [ ]:
STUDENT = "Фамилия Имя Отчество"   # <-- впишите себя

variant = get_variant(STUDENT, lab=1)
describe_variant(variant)

In [ ]:
df_raw, meta = make_table(variant)
TASK, TARGET = meta["task"], meta["target"]

print(f"{meta['domain']}: {df_raw.shape[0]} объектов, {df_raw.shape[1] - 1} признаков")
print(f"тип задачи: {TASK}, целевая переменная: {TARGET}")
df_raw.head(5)

---
# Задача 1. Разведка: что не так с этой таблицей

Повторите на своих данных путь частей 3–4 занятия. Дефекты те же по природе,
что у «Титаника», но есть и два новых, которых там не было: **числовой столбец,
записанный строкой** (типовой экспорт из русского Excel: `'438 900,00'` —
пробел как разделитель разрядов, запятая как десятичный) и **опечатки
масштаба** — потерянный десятичный разделитель, из-за которого отдельные
значения завышены примерно в 100 раз.

### Задание 1.1. Сводка по столбцам

Постройте таблицу: тип, число уникальных значений, доля пропусков, пример
значения. Найдите столбцы, прочитанные как `object`, хотя по смыслу они
числовые, и столбцы, бесполезные уже сейчас (одно значение на всю таблицу).

In [ ]:
# TODO: соберите сводку по столбцам (тип, уникальных, доля пропусков, пример),
#       найдите практически константные столбцы (одно значение чаще 99 %)
#       и посчитайте число полных дубликатов строк.

### Задание 1.2. Числовой столбец, записанный строкой

Напишите `to_numeric_safe(s)`: если столбец объектный, но после удаления
пробелов и замены запятой на точку он разбирается в число — преобразовать.
Порог `threshold`: если не разобралась бóльшая доля непустых значений,
столбец действительно нечисловой, и его надо вернуть без изменений.

In [ ]:
def to_numeric_safe(s: pd.Series, threshold: float = 0.2) -> pd.Series:
    """Строковый столбец -> числовой, если он на самом деле числовой."""
    if s.dtype != object:
        return s

    # TODO (3-4 строки):
    #   1) убрать пробелы (в т.ч. неразрывные \u00a0) и заменить ',' на '.';
    #   2) pd.to_numeric(..., errors="coerce");
    #   3) если доля непреобразованных НЕПУСТЫХ значений > threshold,
    #      столбец действительно нечисловой -- вернуть s без изменений.
    raise NotImplementedError


df = df_raw.copy()
# TODO: примените to_numeric_safe ко всем столбцам и напечатайте, что изменилось.

### Задание 1.3. Категории, дубликаты, опечатки масштаба

Приведите категории к единому написанию (в таблице встречаются `Москва`,
`москва `, `МОСКВА` — для `pandas` это три разные категории), удалите полные
дубликаты и вырожденные признаки, а затем найдите и **исправьте** опечатки
масштаба: значения, превышающие 99-й перцентиль более чем в 30 раз.

Про дубликаты: на занятии мы их **не** удаляли — у «Титаника» нет
идентификатора, и совпадение всех признаков там означало просто двух похожих
пассажиров. Здесь идентификатор есть, и у дублирующихся строк совпадает
**в том числе `id`** — проверьте это сами. Значит, задвоилась запись, а не
объект, и такие строки удаляют.

Для каждого исправленного столбца напечатайте корреляцию с целевой переменной
**до и после** исправления.

In [ ]:
# TODO: 1) привести категории meta["categorical"] к нижнему регистру без пробелов
#          по краям; напечатать число категорий до и после;
#       2) удалить полные дубликаты и вырожденные признаки (одно значение > 99 %);
#       3) найти опечатки масштаба (значение > 30 * квантиль 0.99), поделить их
#          на 100 и напечатать корреляцию признака с целью ДО и ПОСЛЕ.

### Задание 1.4. Признак-утечка

На занятии утечкой был `alive` — копия ответа. В вашей таблице она устроена
тоньше: есть столбец `id`, формально числовой. Обучите дерево **на одном лишь**
`id` и посмотрите на качество; постройте диаграмму рассеяния «`id` против цели».

In [ ]:
# TODO: 1) обучите дерево глубины 4 на ОДНОМ признаке id и оцените качество
#          скользящим контролем (scoring="r2" для регрессии, "roc_auc" для
#          классификации);
#       2) постройте диаграмму рассеяния id против целевой переменной;
#       3) удалите столбец id.

> **Вывод.** Какое качество дал один лишь `id` и откуда взялась эта связь? Чем эта утечка опаснее, чем `alive` с занятия?
>
> *(ваш ответ здесь)*

---
# Задача 2. Заполнение пропусков с учётом механизма

На занятии мы видели, что `dropna()` может выбросить 80 % выборки и сдвинуть
распределение. Теперь разберёмся, чем заполнять — и почему «медианой» это не
всегда правильный ответ.

Мало посчитать долю пропусков — важно понять, **почему** значение отсутствует:

* **MCAR** — пропуск не зависит ни от чего;
* **MAR** — вероятность пропуска зависит от *других наблюдаемых* признаков
  (доход чаще не указывают клиенты с малым стажем);
* **MNAR** — зависит от самого пропущенного значения (не указывают именно
  большие доходы). Худший случай: по данным его не отличить.

В «Титанике» пропуски в `deck` — типичный MAR: палуба известна в основном для
первого класса. Разница практическая: при MAR заполнение общей медианой
систематически искажает признак. Сейчас измерим, насколько.

### Задание 2.1. Найти столбец с MAR и его драйвер

Для каждого числового столбца с пропусками найдите признак, сильнее всего
связанный с *фактом* пропуска. Мера связи: модуль разности средних в группах
«значение есть» и «значение пропущено», в единицах стандартного отклонения.

In [ ]:
num_cols = [c for c in df.select_dtypes(include="number").columns if c != TARGET]
with_na = [c for c in num_cols if df[c].isna().any()]

# TODO: для каждой пары (столбец с пропусками, другой числовой признак)
#       посчитайте |mean(есть) - mean(пропуск)| / std(признака)
#       и соберите таблицу, отсортированную по убыванию этой величины.
link = ...
display(link.head(8).round(3))

# TODO: верхняя строка -- искомая пара
MAR_COL, DRIVER = ..., ...

### Задание 2.2. Два способа заполнения

Заполните пропуски в `MAR_COL` двумя способами — общей медианой и медианой
внутри квартилей драйвера (квартили — квантили уровней 0.25, 0.5 и 0.75, они
делят выборку на четыре равные по численности группы) — и сравните каждый
результат с эталоном: распределением значений на объектах, где значение
известно.

Мера близости — статистика Колмогорова–Смирнова (`scipy.stats.ks_2samp`):
максимальное расхождение эмпирических функций распределения.

In [ ]:
from scipy import stats

known = df.loc[df[MAR_COL].notna(), MAR_COL]
flag = df[MAR_COL].isna()

# TODO: (а) filled_global -- заполнить общей медианой;
#       (б) filled_grouped -- медианой внутри квартилей драйвера
#           (pd.qcut(df[DRIVER], q=4) и groupby(...).transform("median")).
# TODO: для каждого способа посчитайте stats.ks_2samp(заполненные, known)
#       и сведите в таблицу вместе со средним и стандартным отклонением.

In [ ]:
# TODO: постройте гистограммы: известные значения и оба варианта заполнения.

> **Вывод.** Какой способ дал меньшую статистику Колмогорова–Смирнова? Что происходит с дисперсией признака при заполнении константой и чем это опасно?
>
> *(ваш ответ здесь)*

---

> ### ★ Дальше — часть на оценку
>
> Обязательная часть закончилась: если вы дошли досюда и всё работает, работа
> будет принята. Дальше идут задания, по которым выставляется балл. Каждый
> пункт засчитывается отдельно, поэтому имеет смысл сделать хотя бы один.

# Задача 3★. Своя реализация One-Hot кодирования

На занятии мы вызвали `OneHotEncoder` и заметили, что сумма индикаторов по
строке равна единице. Теперь напишем кодировщик сами и доведём это наблюдение
до числа.

### Задание 3.1. Функция `one_hot`

`one_hot(series, categories=None)` возвращает матрицу индикаторов и список
категорий. Если `categories` задан извне, незнакомые значения кодируются
**нулевой строкой** — так ведёт себя `handle_unknown='ignore'`. Это важно:
на контроле может встретиться категория, которой не было в обучении,
а ширина матрицы обязана сохраниться.

In [ ]:
def one_hot(series, categories=None):
    """Индикаторное кодирование. Возвращает (матрица (n, k), список категорий)."""
    # TODO: 1) если categories не задан -- отсортированные уникальные значения без NaN;
    #       2) матрица нулей (n, len(categories));
    #       3) для каждого объекта поставить 1 в столбец своей категории;
    #          незнакомое значение и NaN оставить нулевой строкой.
    raise NotImplementedError

In [ ]:
from sklearn.preprocessing import OneHotEncoder

cat_col = meta["categorical"][0]
known_cat = df[cat_col].dropna()
train, test = known_cat.iloc[: len(known_cat) // 2], known_cat.iloc[len(known_cat) // 2:]

# TODO: закодируйте train (категории из данных) и test (категории из train),
#       сверьте обе матрицы с OneHotEncoder(handle_unknown="ignore"),
#       проверьте, что незнакомое значение и NaN дают нулевые строки.

### Задание 3.2. Зачем `drop='first'`

Сумма всех индикаторов по строке равна единице — то есть совпадает со столбцом
свободного члена. Значит, столбцы линейно зависимы и $X^{\mathsf T}X$ вырождена.

Убедитесь численно: соберите «столбец единиц + полный One-Hot», посмотрите ранг
и число обусловленности, затем повторите с отброшенной первой категорией.
Напомним оценку с занятия: $\mathrm{cond} = 10^{k}$ означает потерю примерно
$k$ верных знаков.

In [ ]:
# TODO: 1) A_full = [столбец единиц | полный One-Hot],
#          A_drop = [столбец единиц | One-Hot без первого столбца];
#       2) для каждой выведите число столбцов, ранг и cond(A^T A);
#       3) покажите, что сумма индикаторов по строке равна 1;
#       4) постройте два РАЗНЫХ вектора весов с одинаковым прогнозом на A_full
#          (прибавьте c к свободному члену и вычтите c из индикаторов).

> **Вывод.** Чему равен ранг в каждом случае и как меняется обусловленность? Почему для решающего дерева этой проблемы не возникает?
>
> *(ваш ответ здесь)*

## Итоги домашней работы

Кратко ответьте на вопросы:

1. Коллега заполнил пропуски средним по всей выборке — включая контрольную часть. Какие два разных дефекта он допустил одновременно?
2. В обучающей выборке признак принимает 40 значений, в контрольной встретилось 41-е. Что произойдёт при `handle_unknown='ignore'` и чем это лучше падения с ошибкой?
3. Сравните утечку `alive` с занятия и утечку `id` из вашей таблицы: какая опаснее на практике и почему?

---

Проверьте перед сдачей: Kernel → Restart & Run All проходит без ошибок,
все ячейки **Вывод** заполнены, графики подписаны.